In [7]:
import json
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [8]:
print("=" * 60)
print("B3 — SÉLECTION DU MODÈLE")
print("Chargement de PEGASUS + test zero-shot sur BookSum")
print("=" * 60)

B3 — SÉLECTION DU MODÈLE
Chargement de PEGASUS + test zero-shot sur BookSum


In [10]:

# ═══════════════════════════════════════════════════════════════
# 1. CHARGEMENT DES CONFIGURATIONS ET DONNÉES
# ═══════════════════════════════════════════════════════════════

# Charger le tokenizer choisi (B2)
with open("tokenizer_choisi.json", "r", encoding="utf-8-sig", errors="replace") as f:
    tokenizer_config = json.load(f)

# Charger les données de Personne A
with open("donnees_propres.json", "r", encoding="utf-8") as f:
    donnees = json.load(f)

MODEL_NAME = "google/pegasus-xsum"
MAX_INPUT_LEN = tokenizer_config["parametres_tokenisation"]["max_input_length"]
MAX_TARGET_LEN = tokenizer_config["parametres_tokenisation"]["max_target_length"]

print(f"\n📌 Configuration:")
print(f"   Modèle cible       : {MODEL_NAME}")
print(f"   Max input length   : {MAX_INPUT_LEN}")
print(f"   Max target length  : {MAX_TARGET_LEN}")
print(f"   Dataset size       : {len(donnees)} exemples")


📌 Configuration:
   Modèle cible       : google/pegasus-xsum
   Max input length   : 512
   Max target length  : 530
   Dataset size       : 100 exemples


In [12]:
# ═══════════════════════════════════════════════════════════════
# 2. CHARGEMENT DU MODÈLE PEGASUS (zero-shot)
# ═══════════════════════════════════════════════════════════════

print("\n🔄 Chargement de PEGASUS (XSUM)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Déplacer sur GPU si dispo
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"   ✅ Modèle chargé sur {device.upper()}")

# Fonction de génération (sans pipeline)
def generate_summary(text, max_length=MAX_TARGET_LEN, min_length=30):
    """Génère un résumé avec PEGASUS"""
    inputs = tokenizer(
        text,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("   ✅ Fonction de génération prête")


🔄 Chargement de PEGASUS (XSUM)...


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 5128.22it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ✅ Modèle chargé sur CPU
   ✅ Fonction de génération prête


In [16]:
# ═══════════════════════════════════════════════════════════════
# 3. TEST ZERO-SHOT (sans pipeline)
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("TEST ZERO-SHOT — Génération sans fine-tuning")
print("=" * 60)

# Définir la fonction de génération
def generate_summary(text, max_length=512, min_length=100):  # ← min_length augmenté
    """Génère un résumé avec PEGASUS"""
    # Troncature correcte
    mots = text.split()
    if len(mots) > MAX_INPUT_LEN:
        text = " ".join(mots[:MAX_INPUT_LEN])
    
    inputs = tokenizer(
        text,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_length=max_length,           # ← 512 tokens max
            min_length=min_length,           # ← 100 tokens min
            num_beams=4,
            early_stopping=True,
            do_sample=False,
            no_repeat_ngram_size=3,          # ← évite les répétitions
            length_penalty=0.8               # ← encourage phrases plus longues
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Tester sur plusieurs exemples
test_indices = [0, 1, 2, 10, 42]
resultats_zero_shot = []

for idx in test_indices:
    exemple = donnees[idx]
    texte_complet = " ".join(exemple["chunks"])
    reference = exemple["resume"]
    
    print(f"\n{'─' * 50}")
    print(f"📖 Exemple {idx}")
    
    try:
        prediction = generate_summary(texte_complet)
        
        resultats_zero_shot.append({
            "index": idx,
            "reference": reference,
            "prediction": prediction,
            "longueur_ref": len(reference.split()),
            "longueur_pred": len(prediction.split())
        })
        
        print(f"   Référence ({len(reference.split())} mots) : {reference[:150]}...")
        print(f"   Prédiction ({len(prediction.split())} mots): {prediction[:150]}...")
        
    except Exception as e:
        print(f"   ❌ Erreur: {str(e)[:100]}")
        resultats_zero_shot.append({
            "index": idx,
            "reference": reference,
            "prediction": f"[ERREUR: {str(e)[:100]}]",
            "longueur_ref": len(reference.split()),
            "longueur_pred": 0
        })


TEST ZERO-SHOT — Génération sans fine-tuning

──────────────────────────────────────────────────
📖 Exemple 0
   Référence (388 mots) : Before any characters appear, the time and geography are made clear. Though it is the last war that England and France waged for a country that neithe...
   Prédiction (102 mots): In our series of letters from African-American journalists, film-maker and columnist Sharmila Tagore considers the role of the wilderness in the colon...

──────────────────────────────────────────────────
📖 Exemple 1
   Référence (198 mots) : In another part of the forest by the river a few miles to the west, Hawkeye and Chingachgook appear to be waiting for someone as they talk with low vo...
   Prédiction (97 mots): This is a quotation from one of Heyward's most famous poems, which he wrote while in the company of a group of men who had been sent to the woods by t...

──────────────────────────────────────────────────
📖 Exemple 2
   Référence (319 mots) : When the mounted 

In [18]:
# ═══════════════════════════════════════════════════════════════
# 4. ÉVALUATION ZERO-SHOT AVEC ROUGE
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("ÉVALUATION ZERO-SHOT — Métriques ROUGE")
print("=" * 60)

try:
    from rouge_score import rouge_scorer
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    scores = {
        "rouge1": [],
        "rouge2": [],
        "rougeL": []
    }
    
    for res in resultats_zero_shot:
        ref = res["reference"]
        pred = res["prediction"]
        if "[ERREUR" not in pred:
            score = scorer.score(ref, pred)
            scores["rouge1"].append(score["rouge1"].fmeasure)
            scores["rouge2"].append(score["rouge2"].fmeasure)
            scores["rougeL"].append(score["rougeL"].fmeasure)
        else:
            scores["rouge1"].append(0.0)
            scores["rouge2"].append(0.0)
            scores["rougeL"].append(0.0)
    
    print("\n📊 ROUGE scores (moyenne sur 5 exemples):")
    print(f"   ROUGE-1 : {np.mean(scores['rouge1']):.3f}")
    print(f"   ROUGE-2 : {np.mean(scores['rouge2']):.3f}")
    print(f"   ROUGE-L : {np.mean(scores['rougeL']):.3f}")
    
    ROUGE_BASELINE = {
        "rouge1": np.mean(scores["rouge1"]),
        "rouge2": np.mean(scores["rouge2"]),
        "rougeL": np.mean(scores["rougeL"])
    }
    
except ImportError:
    print("⚠️ rouge_score non installé. Installation en cours...")
    import subprocess
    subprocess.check_call(["pip", "install", "rouge-score", "-q"])
    from rouge_score import rouge_scorer
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    for res in resultats_zero_shot:
        score = scorer.score(res["reference"], res["prediction"])
        scores["rouge1"].append(score["rouge1"].fmeasure)
        scores["rouge2"].append(score["rouge2"].fmeasure)
        scores["rougeL"].append(score["rougeL"].fmeasure)
    
    print(f"   ROUGE-1 : {np.mean(scores['rouge1']):.3f}")
    print(f"   ROUGE-2 : {np.mean(scores['rouge2']):.3f}")
    print(f"   ROUGE-L : {np.mean(scores['rougeL']):.3f}")
    
    ROUGE_BASELINE = {
        "rouge1": np.mean(scores["rouge1"]),
        "rouge2": np.mean(scores["rouge2"]),
        "rougeL": np.mean(scores["rougeL"])
    }



ÉVALUATION ZERO-SHOT — Métriques ROUGE

📊 ROUGE scores (moyenne sur 5 exemples):
   ROUGE-1 : 0.251
   ROUGE-2 : 0.042
   ROUGE-L : 0.147


In [20]:
# ═══════════════════════════════════════════════════════════════
# 5. COMPARAISON AVEC RÉFÉRENCES (version corrigée)
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("COMPARAISON AVEC RÉFÉRENCES")
print("=" * 60)

REFERENCES = {
    "BART (fine-tuned BookSum)": {"rouge1": 45.2, "rouge2": 22.1, "rougeL": 42.0},
    "PEGASUS (fine-tuned)": {"rouge1": 48.5, "rouge2": 25.3, "rougeL": 45.0},
    "T5 (fine-tuned)": {"rouge1": 44.8, "rouge2": 21.7, "rougeL": 41.5}
}

print("\n📊 Comparaison des performances (ROUGE-1):")
print(f"   {'Modèle':<25} {'ROUGE-1':>10} {'vs zero-shot':>15}")
print(f"   {'-'*25} {'-'*10} {'-'*15}")

baseline_pct = ROUGE_BASELINE["rouge1"] * 100
print(f"   {'PEGASUS (zero-shot)':<25} {baseline_pct:>9.1f}% {'(baseline)':>15}")

for modele, scores in REFERENCES.items():
    gain = scores['rouge1'] - baseline_pct
    gain_pct = (gain / baseline_pct) * 100
    print(f"   {modele:<25} {scores['rouge1']:>9.1f}% {'+':>5}{gain:>5.1f} pts ({gain_pct:>5.0f}%)")


COMPARAISON AVEC RÉFÉRENCES

📊 Comparaison des performances (ROUGE-1):
   Modèle                       ROUGE-1    vs zero-shot
   ------------------------- ---------- ---------------
   PEGASUS (zero-shot)            25.1%      (baseline)
   BART (fine-tuned BookSum)      45.2%     + 20.1 pts (   80%)
   PEGASUS (fine-tuned)           48.5%     + 23.4 pts (   93%)
   T5 (fine-tuned)                44.8%     + 19.7 pts (   79%)


In [21]:

# ═══════════════════════════════════════════════════════════════
# 6. SAUVEGARDE DES RÉSULTATS
# ═══════════════════════════════════════════════════════════════

output = {
    "modele_test": MODEL_NAME,
    "type_test": "zero_shot",
    "nombre_exemples": len(resultats_zero_shot),
    "rouge_scores_baseline": {
        "rouge1": float(ROUGE_BASELINE["rouge1"]),
        "rouge2": float(ROUGE_BASELINE["rouge2"]),
        "rougeL": float(ROUGE_BASELINE["rougeL"])
    },
    "resultats_detailles": [
        {
            "index": r["index"],
            "reference": r["reference"],
            "prediction": r["prediction"],
            "longueur_ref": r["longueur_ref"],
            "longueur_pred": r["longueur_pred"]
        }
        for r in resultats_zero_shot
    ],
    "comparaison_references": REFERENCES,
    "conclusion": {
        "baseline_rouge1": float(ROUGE_BASELINE["rouge1"]),
        "gain_attendu_finetuning": "10-20 points ROUGE-1",
        "modele_recommande": "PEGASUS (meilleures performances sur résumé)",
        "prochaine_etape": "B4 — Fine-tuning avec train_dataset.pt de Personne A"
    }
}

with open("selection_modele_resultats.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("RÉSUMÉ B3")
print("=" * 60)
print(f"✅ Baseline établie : ROUGE-1 = {ROUGE_BASELINE['rouge1']:.3f} ({ROUGE_BASELINE['rouge1']*100:.1f}%)")
print(f"✅ Modèle sélectionné : PEGASUS (google/pegasus-xsum)")
print(f"✅ Résultats sauvegardés : selection_modele_resultats.json")
print("\n📌 Prochaine étape (B4 — Fine-tuning) :")
print("   1. Attendre train_dataset.pt et val_dataset.pt de Personne A")
print("   2. Lancer le fine-tuning de PEGASUS")
print("   3. Sauvegarder model_finetuned/ pour Personne A (Pipeline Map-Reduce)")
print("=" * 60)

# Affichage d'un exemple réussi pour validation
print("\n✨ Exemple de prédiction zero-shot (réussie) :")
exemple_reussi = [r for r in resultats_zero_shot if "[ERREUR" not in r["prediction"]]
if exemple_reussi:
    ex = exemple_reussi[0]
    print(f"\n   📖 Référence  : {ex['reference'][:200]}...")
    print(f"   🤖 Prédiction : {ex['prediction'][:200]}...")


RÉSUMÉ B3
✅ Baseline établie : ROUGE-1 = 0.251 (25.1%)
✅ Modèle sélectionné : PEGASUS (google/pegasus-xsum)
✅ Résultats sauvegardés : selection_modele_resultats.json

📌 Prochaine étape (B4 — Fine-tuning) :
   1. Attendre train_dataset.pt et val_dataset.pt de Personne A
   2. Lancer le fine-tuning de PEGASUS
   3. Sauvegarder model_finetuned/ pour Personne A (Pipeline Map-Reduce)

✨ Exemple de prédiction zero-shot (réussie) :

   📖 Référence  : Before any characters appear, the time and geography are made clear. Though it is the last war that England and France waged for a country that neither would retain, the wilderness between the forces ...
   🤖 Prédiction : In our series of letters from African-American journalists, film-maker and columnist Sharmila Tagore considers the role of the wilderness in the colonial wars of North America, and looks at the role p...
